<a href="https://colab.research.google.com/github/cbonnin88/Python-For-Product/blob/main/EV_Charging_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**The Product:** A network of EV charging stations.

**The Goal:** Understand user engagement, optimize the station experience, and identify areas with high friction (long wait times).

**Core Product Metrics We'll Track:**

- Usage / Engagement: Total charging sessions, Energy consumed.

- User Friction: Waiting time, Queue length.

- Segment Behavior: Performance across different location types (Urban vs. Highway) and vehicle types.

# **Phase 1: Data Cleaning (with Polars)**

In [ ]:
import polars as pl
import gdown as gd
import plotly.express as px

In [ ]:
url = 'https://drive.google.com/uc?id=1cOi_3aElYDe4G4w6Yyu7mve3zG0z41hU'

In [ ]:
file_path = gd.download(url,'charging_ev_and_grid_opitmization_dataset.csv',quiet=True)

In [ ]:
df_ev = pl.read_csv(file_path)

In [ ]:
display(df_ev.head())

timestamp,station_id,location_type,vehicle_id,vehicle_type,arrival_time,charging_start_time,charging_end_time,waiting_time,battery_capacity_kWh,initial_soc,final_soc,energy_consumed_kWh,charging_power_kW,charging_duration,queue_length,station_load,electricity_price,renewable_energy_ratio,traffic_density,weather_condition,day_of_week,time_slot,charging_demand,assigned_charger_id,charging_priority,optimization_reward
str,str,str,str,str,str,str,str,i64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,str,str,str,str,f64,str,str,f64
"""1/1/2025 0:00""","""ST004""","""Urban""","""EV10000""","""Two-Wheeler""","""1/1/2025 0:00""","""1/1/2025 0:12""","""1/1/2025 4:33""",12,60,48.98455,99.865576,30.528616,7,261.67385,4,15.161672,13.66,0.280335,"""Low""","""Cloudy""","""Wednesday""","""Off-Peak""",17.242398,"""CH4""","""Low""",-8.622299
"""1/1/2025 0:15""","""ST005""","""Urban""","""EV10001""","""Two-Wheeler""","""1/1/2025 0:15""","""1/1/2025 0:23""","""1/1/2025 1:12""",8,100,58.495493,100.0,41.504507,50,49.916518,3,20.997219,5.47,0.392127,"""Low""","""Rainy""","""Wednesday""","""Off-Peak""",18.324933,"""CH9""","""Low""",-1.935644
"""1/1/2025 0:30""","""ST019""","""Highway""","""EV10002""","""Car""","""1/1/2025 0:30""","""1/1/2025 0:41""","""1/1/2025 1:35""",11,75,35.711722,95.733464,45.016306,50,54.019568,8,31.606151,9.5,0.103979,"""Low""","""Clear""","""Wednesday""","""Off-Peak""",36.028168,"""CH2""","""Low""",-18.201846
"""1/1/2025 0:45""","""ST008""","""Urban""","""EV10003""","""Two-Wheeler""","""1/1/2025 0:45""","""1/1/2025 0:54""","""1/1/2025 3:29""",9,40,29.270825,100.0,28.29167,11,155.77337,3,21.80305,6.22,0.248553,"""Low""","""Clear""","""Wednesday""","""Off-Peak""",17.146935,"""CH9""","""Medium""",-7.404018
"""1/1/2025 1:00""","""ST008""","""Highway""","""EV10004""","""Two-Wheeler""","""1/1/2025 1:00""","""1/1/2025 1:08""","""1/1/2025 6:14""",8,75,25.585554,100.0,55.810835,11,306.369479,5,15.626266,13.42,0.234926,"""Low""","""Cloudy""","""Wednesday""","""Off-Peak""",14.577768,"""CH1""","""Low""",-6.577466


In [ ]:
print(f'Original Data Shape: {df_ev.shape}')

Original Data Shape: (8354, 27)


In [ ]:
# Data Cleaning & Feature Engineeering

clean_df = (
    df_ev
    .with_columns([
        pl.col('timestamp').str.strptime(pl.Datetime,format='%m/%d/%Y %H:%M',strict=False).alias('timestamp'),
        pl.col('arrival_time').str.strptime(pl.Datetime,format='%m/%d/%Y %H:%M',strict=False).alias('arrival_time'),
        pl.col('charging_start_time').str.strptime(pl.Datetime, format='%m/%d/%Y %H:%M',strict=False).alias('charging_start_time')
    ])
    .with_columns([
        pl.col('timestamp').dt.date().alias('date')
    ])
    .drop_nulls(subset=['timestamp'])
)

display(clean_df.glimpse())

Rows: 8354
Columns: 28
$ timestamp              <datetime[μs]> 2025-01-01 00:00:00, 2025-01-01 00:15:00, 2025-01-01 00:30:00, 2025-01-01 00:45:00, 2025-01-01 01:00:00, 2025-01-01 01:15:00, 2025-01-01 01:30:00, 2025-01-01 01:45:00, 2025-01-01 02:00:00, 2025-01-01 02:15:00
$ station_id                      <str> 'ST004', 'ST005', 'ST019', 'ST008', 'ST008', 'ST014', 'ST004', 'ST020', 'ST004', 'ST020'
$ location_type                   <str> 'Urban', 'Urban', 'Highway', 'Urban', 'Highway', 'Highway', 'Urban', 'Highway', 'Highway', 'Highway'
$ vehicle_id                      <str> 'EV10000', 'EV10001', 'EV10002', 'EV10003', 'EV10004', 'EV10005', 'EV10006', 'EV10007', 'EV10008', 'EV10009'
$ vehicle_type                    <str> 'Two-Wheeler', 'Two-Wheeler', 'Car', 'Two-Wheeler', 'Two-Wheeler', 'Bus', 'Bus', 'Car', 'Car', 'Two-Wheeler'
$ arrival_time           <datetime[μs]> 2025-01-01 00:00:00, 2025-01-01 00:15:00, 2025-01-01 00:30:00, 2025-01-01 00:45:00, 2025-01-01 01:00:00, 2025-01-01 01:1

None

In [ ]:
display(clean_df.head())

timestamp,station_id,location_type,vehicle_id,vehicle_type,arrival_time,charging_start_time,charging_end_time,waiting_time,battery_capacity_kWh,initial_soc,final_soc,energy_consumed_kWh,charging_power_kW,charging_duration,queue_length,station_load,electricity_price,renewable_energy_ratio,traffic_density,weather_condition,day_of_week,time_slot,charging_demand,assigned_charger_id,charging_priority,optimization_reward,date
datetime[μs],str,str,str,str,datetime[μs],datetime[μs],str,i64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,str,str,str,str,f64,str,str,f64,date
2025-01-01 00:00:00,"""ST004""","""Urban""","""EV10000""","""Two-Wheeler""",2025-01-01 00:00:00,2025-01-01 00:12:00,"""1/1/2025 4:33""",12,60,48.98455,99.865576,30.528616,7,261.67385,4,15.161672,13.66,0.280335,"""Low""","""Cloudy""","""Wednesday""","""Off-Peak""",17.242398,"""CH4""","""Low""",-8.622299,2025-01-01
2025-01-01 00:15:00,"""ST005""","""Urban""","""EV10001""","""Two-Wheeler""",2025-01-01 00:15:00,2025-01-01 00:23:00,"""1/1/2025 1:12""",8,100,58.495493,100.0,41.504507,50,49.916518,3,20.997219,5.47,0.392127,"""Low""","""Rainy""","""Wednesday""","""Off-Peak""",18.324933,"""CH9""","""Low""",-1.935644,2025-01-01
2025-01-01 00:30:00,"""ST019""","""Highway""","""EV10002""","""Car""",2025-01-01 00:30:00,2025-01-01 00:41:00,"""1/1/2025 1:35""",11,75,35.711722,95.733464,45.016306,50,54.019568,8,31.606151,9.5,0.103979,"""Low""","""Clear""","""Wednesday""","""Off-Peak""",36.028168,"""CH2""","""Low""",-18.201846,2025-01-01
2025-01-01 00:45:00,"""ST008""","""Urban""","""EV10003""","""Two-Wheeler""",2025-01-01 00:45:00,2025-01-01 00:54:00,"""1/1/2025 3:29""",9,40,29.270825,100.0,28.29167,11,155.77337,3,21.80305,6.22,0.248553,"""Low""","""Clear""","""Wednesday""","""Off-Peak""",17.146935,"""CH9""","""Medium""",-7.404018,2025-01-01
2025-01-01 01:00:00,"""ST008""","""Highway""","""EV10004""","""Two-Wheeler""",2025-01-01 01:00:00,2025-01-01 01:08:00,"""1/1/2025 6:14""",8,75,25.585554,100.0,55.810835,11,306.369479,5,15.626266,13.42,0.234926,"""Low""","""Cloudy""","""Wednesday""","""Off-Peak""",14.577768,"""CH1""","""Low""",-6.577466,2025-01-01


# **Phase 2: Exploratory Data Analysis (EDA)**

1. Engagement Metric: Daily Charging Sessions
- Are peoplle using our charging stations ?

In [ ]:
# Aggregate data using Polars: Count sessions per day
daily_usage = (
    clean_df
    .group_by('date')
    .agg(pl.len().alias('total_sessions'))
    .sort('date')
)

display(daily_usage.head())

date,total_sessions
date,u32
2025-01-01,96
2025-01-02,96
2025-01-03,96
2025-01-04,96
2025-01-05,96


In [ ]:
fig_daily_usage = px.line(
    daily_usage.to_pandas(),
    x='date',
    y='total_sessions',
    title='Product Engagement: Daily Charging Sessions',
    labels={'date':'Date','total_sessions':'Number of Sessions'},
    markers=True
)

fig_daily_usage.show()

2. User Friction: Average Waiting Time by Location
- Waiting time is a massive friction point in the EV user journey. If users wait too long at Urban stations vs Highway stations, we might need to build more chargers in the city.

In [ ]:
friction_df = (
    clean_df
    .group_by('location_type')
    .agg(pl.col('waiting_time').mean().alias('avg_waiting_time_mins'))
    .sort('avg_waiting_time_mins',descending=True)
)

display(friction_df)

location_type,avg_waiting_time_mins
str,f64
"""Highway""",9.598896
"""Urban""",9.454155


In [ ]:
fig_friction = px.bar(
    friction_df.to_pandas(),
    x='location_type',
    y='avg_waiting_time_mins',
    color='location_type',
    title='User Friction: Average Waiting Time by Location Type',
    labels={'location_type':'Location Type','avg_waiting_time_mins':'Average Wait Time (Minutes)'},
    color_discrete_sequence=px.colors.sequential.Viridis_r
)

fig_friction.show()

3. Value Delivered: Energy Consumed by Vehicule Type
- Different user personas (Buses vs. Cars) extract different value from our product.

In [ ]:
fig_energy_consumed = px.box(
    clean_df.to_pandas(),
    x='vehicle_type',
    y='energy_consumed_kWh',
    color='vehicle_type',
    title='Value Delivered: Energy Consumed per Sessions by Vehicle Type',
    labels = {'vehicle_type':'Vehicle Types','energy_consumed_kWh':'Energy Consumed (kWh)'}
)

fig_energy_consumed.show()

In [ ]:
clean_df.write_csv('clean_ev_charging_data.csv')
print('Clean data exported successfullly for Looker (Data) Studio')

Clean data exported successfullly for Looker (Data) Studio
